In [ ]:
# ---------------- Imports ----------------
import json
import os
from collections import defaultdict

import pandas as pd
import yaml
from sklearn.metrics import confusion_matrix, recall_score
import numpy as np


import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl

# Path to font
FONT_DIR = os.path.join("../../config", "fonts", "linux_libertine")
FONT_PATH = os.path.join(FONT_DIR, "LinLibertine_R.ttf")

# Register font
fm.fontManager.addfont(FONT_PATH)

# Get font name
libertine_font = fm.FontProperties(fname=FONT_PATH).get_name()

# Set globally
mpl.rcParams.update({
    "font.family": libertine_font,
    "pdf.fonttype": 42,
})
    


In [ ]:
# ---------------- Args ----------------


# Llama 3.1 8B @ 0.3
output_file_name = "llama-3.1-8b-instruct-combined-claims-15k-0.3-100x100trajs"
RESULTS_FILES = {
    "baseline": [
        "20260202t123134-20260128T2129-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1-100x100trajs",
        "20260202t144219-20260130T1707-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1-100x100trajs",
        "20260202t150350-20260130T1730-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1-100x100trajs",
    ],
    "authoritative": [
        "20260202t125312-20260130T1258-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3-100x100trajs",
        "20260202t152524-20260130T1617-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3-100x100trajs",
        "20260202t154657-20260130T1641-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3-100x100trajs",
    ],
    "consensus": [
        "20260202t131457-20260131T1055-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3-100x100trajs",
        "20260202t160847-20260131T1118-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3-100x100trajs",
        "20260202t163026-20260131T1141-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3-100x100trajs",
    ],
    "emotional": [
        "20260202t135819-20260131T1314-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3-100x100trajs",
        "20260202t165206-20260131T1336-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3-100x100trajs",
        "20260202t171342-20260131T1359-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3-100x100trajs",
    ],
    "prestige": [
        "20260202t133635-20260131T1206-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3-100x100trajs",
        "20260202t173510-20260131T1229-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3-100x100trajs",
        "20260202t175657-20260131T1251-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3-100x100trajs",
    ],
    "sensationalist": [
        "20260202t141948-20260131T1421-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3-100x100trajs",
        "20260202t181833-20260131T1444-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3-100x100trajs",
        "20260202t184008-20260131T1506-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3-100x100trajs",
    ],
}




In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]
RESULTS_FOLDER = os.path.join(PROJ_STORE, "experiments", "model-evaluate-trajectory")


# OUTPUT
OUTPUT_DIR = os.path.join(PROJ_STORE, "evaluation", "trajectories-results")
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"{output_file_name}")
                          
                          



In [ ]:
# -------------------------
# Functions
# -------------------------

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)




In [ ]:
all_rows = []

for model_target, fnames in RESULTS_FILES.items():

    for fname in fnames:

        path = os.path.join(RESULTS_FOLDER, f"{fname}.jsonl")

        for row in load_jsonl(path):

            row["model_framing_target"] = model_target
            row["run_id"] = fname   # track run

            all_rows.append(row)


all_df = pd.DataFrame(all_rows)

display(all_df.head())
display(all_df.shape)

In [ ]:
# -------------------------
# Load baseline (benchmark)
# -------------------------

orig_df = all_df[all_df["model_framing_target"] == "baseline"]


display(orig_df.head())
print(orig_df.shape)




In [ ]:

orig_traj = (
    orig_df
    .groupby(["run_id", "trajectory_id"])
    .agg(
        T=("step", "max"),
        auc_H=("H_t", "sum"),
    )
    .reset_index()
)

orig_traj["mean_auc_H"] = orig_traj["auc_H"] / orig_traj["T"]

benchmark = {
    "p25": np.percentile(orig_traj["mean_auc_H"], 25),
    "p50": np.percentile(orig_traj["mean_auc_H"], 50),
    "p75": np.percentile(orig_traj["mean_auc_H"], 75),
    "mean": orig_traj["mean_auc_H"].mean(),
    "std": orig_traj["mean_auc_H"].std(),
}

display(benchmark)

In [ ]:
# ---------------- Plot ----------------
step_summary = (
    all_df
    .groupby(["model_framing_target", "run_id", "step"])["H_t"]
    .mean()
    .reset_index()
)

# now average across runs
step_summary = (
    step_summary
    .groupby(["model_framing_target", "step"])["H_t"]
    .mean()
    .reset_index()
)

# avoid divide-by-zero
step_summary = step_summary.query("step > 0")
step_summary["H_per_step"] = step_summary["H_t"] / step_summary["step"]


# Desired order (put "baseline" first, then others)
ORDER = ["baseline", "authoritative", "consensus", "emotional", "prestige", "sensationalist"]

# Build mapping: lowercase -> display name
LABEL_MAP = {
    k: k.capitalize() for k in ORDER
}

plt.figure(figsize=(7, 5))


cmap = plt.get_cmap("tab10")

colors = cmap.colors

for i, key in enumerate(ORDER):

    g = step_summary[step_summary["model_framing_target"] == key]

    linestyle = "--" if key == "baseline" else "-"

    plt.plot(
        g["step"],
        g["H_per_step"],
        label=LABEL_MAP[key],
        color=colors[i],
        linestyle=linestyle,
        linewidth=2
    )




plt.xlabel("Step $t$", fontsize=22)
plt.ylabel("IH", fontsize=22)

#plt.title("Average Information Health per Step", fontsize=22)

plt.legend(loc="upper right", fontsize=20)

plt.xticks(fontsize=22)
plt.yticks(fontsize=22)



plt.tight_layout()
plt.savefig(f"{OUTPUT_FILE}-ih-average.pdf", format="pdf", bbox_inches="tight")
plt.show()



In [ ]:
# ---------------- Cumulative H_t plot ----------------

cum_summary = (
    all_df
    .groupby(["model_framing_target", "run_id", "step"])["H_t"]
    .mean()
    .reset_index()
)

cum_summary = (
    cum_summary
    .groupby(["model_framing_target", "step"])["H_t"]
    .mean()
    .reset_index()
)

plt.figure()

for model, g in cum_summary.groupby("model_framing_target"):
    g = g.sort_values("step")
    plt.plot(g["step"], g["H_t"], label=model)

plt.xlabel("Step $t$")
plt.ylabel(r"Cumulative Information Health")
plt.legend(title="Model framing target")
plt.tight_layout()
plt.savefig(f"{OUTPUT_FILE}-ih-cumulative.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
# -------------------------------------------------
# Extract per-step confidence
# -------------------------------------------------

df_all = all_df.copy()

# correctness
df_all["is_correct"] = df_all["predicted_label"] == df_all["true_label"]

# model confidence
df_all["confidence"] = df_all.apply(
    lambda r: r["scores"][r["predicted_label"]]["word_cond_prob"],
    axis=1
)

# split confidence by correctness (precompute to avoid leakage)
df_all["conf_correct"] = np.where(
    df_all["is_correct"],
    df_all["confidence"],
    np.nan
)

df_all["conf_incorrect"] = np.where(
    ~df_all["is_correct"],
    df_all["confidence"],
    np.nan
)


# -------------------------------------------------
# Per-run aggregation
# -------------------------------------------------

per_run = (
    df_all
    .groupby(["model_framing_target", "run_id"])
    .agg(
        n_supports_encountered=("true_label", lambda x: (x == "SUPPORTS").sum()),
        n_refutes_encountered=("true_label", lambda x: (x == "REFUTES").sum()),

        P_correct=("is_correct", "mean"),

        mean_conf_correct=("conf_correct", "mean"),
        mean_conf_incorrect=("conf_incorrect", "mean"),
    )
    .reset_index()
)


# -------------------------------------------------
# Average across runs
# -------------------------------------------------

summary = (
    per_run
    .groupby("model_framing_target", as_index=False)
    .mean(numeric_only=True)
)


# derived metrics
summary["P_incorrect"] = 1.0 - summary["P_correct"]

summary["confidence_gap"] = (
    summary["mean_conf_correct"]
    - summary["mean_conf_incorrect"]
)


# -------------------------------------------------
# Output
# -------------------------------------------------

display(summary)

summary.to_csv(f"{OUTPUT_FILE}-summary.csv", index=False)


In [ ]:
## Trajectory summaries for framings only

traj_summary = (
    all_df
    .groupby(["model_framing_target", "run_id", "trajectory_id"])
    .agg(
        T=("step", "max"),
        auc_H=("H_t", "sum"),
    )
    .reset_index()
)

traj_summary["mean_auc_H"] = traj_summary["auc_H"] / traj_summary["T"]



display(traj_summary.shape)
display(traj_summary.head())


In [ ]:
mean_IH = (
    traj_summary
    .groupby(["model_framing_target", "run_id"])["mean_auc_H"]
    .mean()
    .reset_index()
)

mean_IH = (
    mean_IH
    .groupby("model_framing_target")["mean_auc_H"]
    .mean()
    .reset_index(name="mean_IH")
)

display(mean_IH)

mean_IH.to_csv(f"{OUTPUT_FILE}-mean-ih.csv", index=False)
